In [1]:
from ibkr_connector import IBKRConnector
from ibapi.contract import Contract
from vol_math import VolMath
from visualizer import VolVisualizer
import pandas as pd

connector = IBKRConnector()
# Connect to TWS (ensure TWS is open and API enabled)
connected = connector.connect_tws(port=7496) 
print(f"Connected: {connected}")

Connected: True


In [2]:
from datetime import datetime
import pandas as pd
import time

# --- 1. CONFIGURATION ---
FUTURE_CONID = 602619718  # Your confirmed July 2026 Corn Future ID
OPTION_EXPIRY = "202606"   # The Options cycle for July delivery
R_RATE = 0.045             # 4.5% Risk-free rate
STRIKE_STEP = 10           # We'll look at strikes every 10 cents

# --- 2. SETUP CONTRACTS ---
# Define the Future using the ID (Bulletproof method)
future_contract = Contract()
future_contract.conId = FUTURE_CONID
future_contract.exchange = "CBOT"

# 3. PULL LIVE UNDERLYING PRICE
print(f"Requesting live price for Future ID: {FUTURE_CONID}...")
underlying_price = connector.get_corn_price(FUTURE_MONTH_STR := "202607") 

if underlying_price is None or underlying_price <= 0:
    print("Error: Could not get live price. Check TWS permissions for CBOT.")
else:
    print(f"Current July Corn Future: {underlying_price} cents")

    # --- 4. CALCULATE TIME TO EXPIRY (T) ---
    # July Options expire June 26, 2026
    today = datetime(2026, 1, 22)
    expiry_date = datetime(2026, 6, 26) 
    T = (expiry_date - today).days / 365.0
    print(f"Years to Expiry: {T:.4f}")

    # --- 5. THE OPTION SCANNER ---
    # We define a range of strikes around the current price (ATM +/- 50 cents)
    atm_strike = round(underlying_price / 10) * 10
    strikes_to_scan = range(atm_strike - 50, atm_strike + 60, STRIKE_STEP)
    
    market_data = []
    print("\nScanning Option Chain for Implied Volatilities...")
    
    for strike in strikes_to_scan:
        # Request price for each Call option
        # The connector handles the 'Messenger' work here
        opt_price = connector.get_option_market_price(OPTION_EXPIRY, float(strike), "C")
        
        if opt_price and opt_price > 0:
            # Solve for IV using the VolMath engine
            iv = VolMath.solve_iv(
                market_price=opt_price,
                S=underlying_price,
                K=strike,
                T=T,
                r=R_RATE,
                option_type="C"
            )
            
            if not pd.isna(iv):
                print(f"  Strike {strike}: Price={opt_price:<6} | IV={iv:.2%}")
                market_data.append({
                    'strike': strike, 
                    'iv': iv, 
                    'time_to_expiry': T,
                    'market_price': opt_price
                })
        else:
            print(f"  Strike {strike}: No liquid price found.")

    # --- 6. PREPARE FOR VISUALIZATION ---
    if market_data:
        df = pd.DataFrame(market_data)
        print(f"\nSuccessfully captured {len(df)} points for the Vol Surface.")
        
        # This sends the data to your 3D visualizer
        s_m, t_m, i_m = VolMath.construct_surface(df)
        fig = VolVisualizer.plot_3d(s_m, t_m, i_m)
        fig.show()
    else:
        print("No market data points collected. Surface cannot be generated.")

Requesting live price for Future ID: 602619718...
IBKR Error 200: The destination or exchange selected is Invalid. Please review your order's "Destination" field. If using a <br>Directed order, review the exchange selected when creating the order ticket or order row. This may occur when <br>creating stock orders for the overnight session or when creating option orders for the overnight session.
Error: Could not get live price. Check TWS permissions for CBOT.


IBKR Error 2103: Market data farm connection is broken:usfarm
